# YouTube Channel Data Analysis

## Project Overview
This notebook analyzes YouTube channel data to answer three key questions:
1. **Which channels/categories grow fastest?**
2. **Does upload frequency affect views?**
3. **Which category gets the most engagement?**

**Dataset**: YouTube video statistics and comments data

**Date**: March 2026

## 1. Import Required Libraries

In [ ]:
# Import necessary libraries
import sys
import os
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Import custom modules
from cleaning import load_data, clean_videos_data, clean_comments_data, merge_data
from analysis import (analyze_channel_growth, analyze_upload_frequency_vs_views, 
                     analyze_category_engagement, get_top_performing_videos,
                     analyze_sentiment_by_category, create_summary_statistics)

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ All libraries imported successfully!")

## 2. Load and Explore the Dataset

In [ ]:
# Load the raw data
print("Loading data...")
videos_df, comments_df = load_data()

print(f"\n📊 Dataset Overview:")
print(f"  - Total videos: {len(videos_df)}")
print(f"  - Total comments: {len(comments_df)}")

print("\n📹 Videos Data - First 5 rows:")
display(videos_df.head())

print("\n💬 Comments Data - First 5 rows:")
display(comments_df.head())

print("\n📋 Videos Data Info:")
print(videos_df.info())

print("\n📋 Videos Data Summary Statistics:")
display(videos_df.describe())

## 3. Data Cleaning and Preprocessing

In [ ]:
# Clean the data
print("Cleaning videos data...")
videos_clean = clean_videos_data(videos_df)

print("Cleaning comments data...")
comments_clean = clean_comments_data(comments_df)

print("\n✓ Data cleaning complete!")
print(f"\n📊 Cleaned Data Overview:")
print(f"  - Videos: {len(videos_clean)} rows")
print(f"  - Comments: {len(comments_clean)} rows")
print(f"  - Date range: {videos_clean['Published At'].min()} to {videos_clean['Published At'].max()}")
print(f"  - Categories: {videos_clean['Keyword'].nunique()}")

print("\n📹 New Columns Added to Videos Data:")
new_cols = [col for col in videos_clean.columns if col not in videos_df.columns]
for col in new_cols:
    print(f"  - {col}")

display(videos_clean.head())

## 4. Question 1: Which Channels/Categories Grow Fastest?

We'll analyze channel growth by examining:
- Total views and engagement metrics
- Growth scores based on multiple factors
- Average performance per video

In [ ]:
# Analyze channel growth
growth_analysis = analyze_channel_growth(videos_clean)

print("🚀 CHANNEL GROWTH ANALYSIS")
print("="*70)
display(growth_analysis)

print("\n📊 Top 5 Fastest Growing Categories:")
top_5_growth = growth_analysis.head(5)
display(top_5_growth[['Keyword', 'Total_Views', 'Avg_Views_Per_Video', 
                       'Avg_Engagement_Rate', 'Growth_Score']])

## 5. Question 2: Does Upload Frequency Affect Views?

We'll analyze the correlation between:
- Number of videos uploaded per time period
- Average views received
- Correlation coefficients by category

In [ ]:
# Analyze upload frequency vs views
freq_correlation, freq_data = analyze_upload_frequency_vs_views(videos_clean)

print("📈 UPLOAD FREQUENCY vs VIEWS ANALYSIS")
print("="*70)
print("\nCorrelation between Upload Frequency and Views by Category:")
display(freq_correlation)

print("\n🔍 Key Insights:")
for idx, row in freq_correlation.iterrows():
    corr = row['Correlation_Freq_Views']
    if pd.notna(corr):
        category = row['Category']
        avg_freq = row['Avg_Upload_Frequency']
        avg_views = row['Avg_Views']
        
        if corr > 0.3:
            relationship = "POSITIVE correlation"
        elif corr < -0.3:
            relationship = "NEGATIVE correlation"
        else:
            relationship = "WEAK/NO correlation"
            
        print(f"  - {category.upper()}: {relationship} (r={corr:.2f})")
        print(f"    Avg upload frequency: {avg_freq:.2f} videos/period, Avg views: {avg_views:,.0f}")

## 6. Question 3: Which Category Gets the Most Engagement?

We'll examine engagement metrics including:
- Engagement rate (likes + comments per view)
- Like rate
- Comment rate
- Overall engagement score

In [ ]:
# Analyze category engagement
engagement_analysis = analyze_category_engagement(videos_clean)

print("💬 CATEGORY ENGAGEMENT ANALYSIS")
print("="*70)
display(engagement_analysis)

print("\n🏆 Top 5 Most Engaging Categories:")
top_5_engagement = engagement_analysis.head(5)
display(top_5_engagement[['Keyword', 'Avg_Views', 'Avg_Likes', 'Avg_Comments',
                          'Avg_Engagement_Rate', 'Engagement_Score']])

print("\n📊 Winner: Highest Overall Engagement")
winner = engagement_analysis.iloc[0]
print(f"  Category: {winner['Keyword'].upper()}")
print(f"  Engagement Score: {winner['Engagement_Score']:.2f}")
print(f"  Avg Engagement Rate: {winner['Avg_Engagement_Rate']:.2f}%")
print(f"  Avg Like Rate: {winner['Avg_Like_Rate']:.2f}%")
print(f"  Avg Comment Rate: {winner['Avg_Comment_Rate']:.2f}%")

## 7. Visualizations

Visual representations of our key findings

In [ ]:
# Visualization 1: Channel Growth Comparison
fig, ax = plt.subplots(figsize=(14, 8))

top_10_growth = growth_analysis.head(10)
x_pos = np.arange(len(top_10_growth))

ax.barh(x_pos, top_10_growth['Growth_Score'], color='#4CAF50', alpha=0.8)
ax.set_yticks(x_pos)
ax.set_yticklabels(top_10_growth['Keyword'].str.upper())
ax.set_xlabel('Growth Score', fontsize=12, fontweight='bold')
ax.set_title('Top 10 Fastest Growing Categories', fontsize=16, fontweight='bold', pad=20)
ax.grid(axis='x', alpha=0.3)

# Add value labels
for i, v in enumerate(top_10_growth['Growth_Score']):
    ax.text(v + 50000, i, f'{v:,.0f}', va='center', fontsize=10)

plt.tight_layout()
plt.savefig('../visuals/channel_growth.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Chart saved to visuals/channel_growth.png")

In [ ]:
# Visualization 2: Upload Frequency vs Views Scatter Plot
fig, ax = plt.subplots(figsize=(14, 8))

categories = freq_data['Category'].unique()
colors = sns.color_palette('husl', len(categories))

for i, category in enumerate(categories):
    cat_data = freq_data[freq_data['Category'] == category]
    ax.scatter(cat_data['Videos_Uploaded'], cat_data['Avg_Views'], 
              label=category.upper(), alpha=0.7, s=100, color=colors[i])

ax.set_xlabel('Videos Uploaded per Period', fontsize=12, fontweight='bold')
ax.set_ylabel('Average Views', fontsize=12, fontweight='bold')
ax.set_title('Upload Frequency vs Average Views by Category', fontsize=16, fontweight='bold', pad=20)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../visuals/upload_frequency_vs_views.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Chart saved to visuals/upload_frequency_vs_views.png")

In [ ]:
# Visualization 3: Category Engagement Heatmap
fig, ax = plt.subplots(figsize=(12, 8))

# Prepare data for heatmap
engagement_viz = engagement_analysis.set_index('Keyword')[
    ['Avg_Engagement_Rate', 'Avg_Like_Rate', 'Avg_Comment_Rate']
].head(10)

sns.heatmap(engagement_viz, annot=True, fmt='.2f', cmap='YlOrRd', 
            cbar_kws={'label': 'Rate (%)'}, ax=ax, linewidths=0.5)

ax.set_title('Engagement Metrics by Category (Top 10)', fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('Engagement Metrics', fontsize=12, fontweight='bold')
ax.set_ylabel('Category', fontsize=12, fontweight='bold')
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)

plt.tight_layout()
plt.savefig('../visuals/category_engagement_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Chart saved to visuals/category_engagement_heatmap.png")

In [ ]:
# Visualization 4: Comprehensive Dashboard - Top Categories
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('YouTube Channel Analysis Dashboard', fontsize=20, fontweight='bold', y=0.995)

# Subplot 1: Total Views by Category
top_views = videos_clean.groupby('Keyword')['Views'].sum().sort_values(ascending=False).head(8)
axes[0, 0].barh(range(len(top_views)), top_views.values, color='#2196F3')
axes[0, 0].set_yticks(range(len(top_views)))
axes[0, 0].set_yticklabels(top_views.index.str.upper())
axes[0, 0].set_xlabel('Total Views')
axes[0, 0].set_title('Total Views by Category', fontweight='bold')
axes[0, 0].grid(axis='x', alpha=0.3)

# Subplot 2: Average Engagement Rate
top_engagement = engagement_analysis.head(8)
axes[0, 1].bar(range(len(top_engagement)), top_engagement['Avg_Engagement_Rate'], color='#FF9800')
axes[0, 1].set_xticks(range(len(top_engagement)))
axes[0, 1].set_xticklabels(top_engagement['Keyword'].str.upper(), rotation=45, ha='right')
axes[0, 1].set_ylabel('Engagement Rate (%)')
axes[0, 1].set_title('Average Engagement Rate by Category', fontweight='bold')
axes[0, 1].grid(axis='y', alpha=0.3)

# Subplot 3: Video Count by Category
video_count = videos_clean['Keyword'].value_counts().head(8)
axes[1, 0].pie(video_count.values, labels=video_count.index.str.upper(), autopct='%1.1f%%', startangle=90)
axes[1, 0].set_title('Video Distribution by Category', fontweight='bold')

# Subplot 4: Views vs Engagement Scatter
category_stats = videos_clean.groupby('Keyword').agg({
    'Views': 'mean',
    'Engagement_Rate': 'mean'
}).reset_index()

scatter = axes[1, 1].scatter(category_stats['Views'], category_stats['Engagement_Rate'], 
                             s=200, alpha=0.6, c=range(len(category_stats)), cmap='viridis')
axes[1, 1].set_xlabel('Average Views')
axes[1, 1].set_ylabel('Average Engagement Rate (%)')
axes[1, 1].set_title('Views vs Engagement Rate', fontweight='bold')
axes[1, 1].grid(alpha=0.3)

# Add category labels to scatter plot
for idx, row in category_stats.iterrows():
    axes[1, 1].annotate(row['Keyword'].upper()[:4], 
                       (row['Views'], row['Engagement_Rate']),
                       fontsize=8, alpha=0.7)

plt.tight_layout()
plt.savefig('../visuals/comprehensive_dashboard.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Chart saved to visuals/comprehensive_dashboard.png")

## 8. Summary & Conclusions

### Key Findings

In [ ]:
# Generate comprehensive summary
summary = create_summary_statistics(videos_clean)

print("="*70)
print("📊 YOUTUBE CHANNEL DATA ANALYSIS - EXECUTIVE SUMMARY")
print("="*70)

print("\n📈 DATASET OVERVIEW:")
print(f"  • Total Videos Analyzed: {summary['total_videos']:,}")
print(f"  • Total Views: {summary['total_views']:,.0f}")
print(f"  • Total Likes: {summary['total_likes']:,.0f}")
print(f"  • Total Comments: {summary['total_comments']:,.0f}")
print(f"  • Number of Categories: {summary['categories_count']}")
print(f"  • Date Range: {summary['date_range']}")

print("\n🚀 QUESTION 1: Which channels/categories grow fastest?")
top_growth = growth_analysis.iloc[0]
print(f"  ANSWER: {top_growth['Keyword'].upper()}")
print(f"  • Growth Score: {top_growth['Growth_Score']:,.0f}")
print(f"  • Total Views: {top_growth['Total_Views']:,.0f}")
print(f"  • Avg Views per Video: {top_growth['Avg_Views_Per_Video']:,.0f}")
print(f"  • Avg Engagement Rate: {top_growth['Avg_Engagement_Rate']:.2f}%")

print("\n📊 QUESTION 2: Does upload frequency affect views?")
print(f"  ANSWER: Correlation varies by category")
for idx, row in freq_correlation.head(3).iterrows():
    if pd.notna(row['Correlation_Freq_Views']):
        print(f"  • {row['Category'].upper()}: Correlation = {row['Correlation_Freq_Views']:.2f}")

print("\n💬 QUESTION 3: Which category gets the most engagement?")
top_eng = engagement_analysis.iloc[0]
print(f"  ANSWER: {top_eng['Keyword'].upper()}")
print(f"  • Engagement Score: {top_eng['Engagement_Score']:.2f}")
print(f"  • Avg Engagement Rate: {top_eng['Avg_Engagement_Rate']:.2f}%")
print(f"  • Avg Like Rate: {top_eng['Avg_Like_Rate']:.2f}%")
print(f"  • Avg Comment Rate: {top_eng['Avg_Comment_Rate']:.2f}%")

print("\n🏆 TOP 3 PERFORMING VIDEOS:")
top_videos = get_top_performing_videos(videos_clean, n=3)
for i, (idx, video) in enumerate(top_videos.iterrows(), 1):
    print(f"\n  {i}. {video['Title'][:60]}...")
    print(f"     Category: {video['Keyword'].upper()} | Views: {video['Views']:,.0f} | Likes: {video['Likes']:,.0f}")

print("\n" + "="*70)
print("✓ Analysis Complete!")